In [ ]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/pseudobulk_test/yao_2021/ACA_MOp_VISp")

library(dplyr)
library(data.table)

source("/mnt/lareaulab/reliscu/code/FindModules/FindModules.R")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


Loading required package: dynamicTreeCut

Loading required package: fastcluster


Attaching package: ‘fastcluster’


The following object is masked from ‘package:stats’:

    hclust





Attaching package: ‘WGCNA’


The following object is masked from ‘package:stats’:

    cor



Attaching package: ‘flashClust’


The following object is masked from ‘package:fastcluster’:

    hclust


The following object is masked from ‘package:stats’:

    hclust



Attaching package: ‘svMisc’


The following object is masked from ‘package:utils’:

    ?


Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:dplyr’:



Allowing multi-threading with up to 48 threads.


Here I run FM to (hopefully) find modules representing each of the cell types present in the single-cell data

In [ ]:
counts <- fread("data/SyntheticDatasets/SyntheticDataset1_10pcntCells_5SD_200samples_02-36-30.csv", data.table=FALSE)

# For duplicate genes, choose row with highest mean expression

mean_expr <- data.frame(
    Index=1:nrow(counts), 
    Gene=counts[,1], 
    Expr=rowMeans(counts[,-1])
)

mean_expr <- mean_expr %>%
    group_by(Gene) %>%
    slice_max(Expr)

print(dim(mean_expr))

counts <- counts[mean_expr$Index,]

[1] 38029     3


In [ ]:
# Remove 0 expression genes
counts_filtered <- counts[rowSums(counts[,-1]) > 0,]
dim(counts_filtered)

In [ ]:
# Subset to genes in the top X percentile for generating modules

prob <- .5
mean_expr <- rowMeans(counts_filtered[,-1])

subset_cutoff <- unname(quantile(mean_expr, prob))
print(paste("quantile(mean_expr, prob):", round(subset_cutoff, 2)))
subset <- mean_expr >= subset_cutoff
sum(subset)

In [ ]:
# Normalize counts by sample

total_expr <- colSums(counts_filtered[,-1])
expr <- sweep(counts_filtered[,-1], MARGIN=2, FUN="/", STATS=total_expr) * 1e4
# expr <- log2(expr + 1)
expr <- data.frame(Gene=counts_filtered[,1], expr)

In [ ]:
merge.param <- 0.95
projectname <- paste0("yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_5SD_200samples_mergeParam", merge.param, "_subsetCutoff", round(subset_cutoff, 2))

In [ ]:
FindModules(
  projectname=projectname,
  expr=expr,
  geneinfo=1,
  sampleindex=2:ncol(expr),
  samplegroups=NULL,
  subset=subset,
  simMat=NULL,
  saveSimMat=FALSE,
  simType="Bicor",
  beta=1,
  overlapType="None",
  TOtype="signed",
  TOdenom="min",
  MIestimator="mi.mm",
  MIdisc="equalfreq",
  signumType="rel",
  iterate=TRUE,
  signumvec=c(.9999,.999,.99,.98,.97,.96,.95),
  minsizevec=c(3,4,6,8,10),
  signum=NULL,
  minSize=NULL,
  merge.by="ME",
  merge.param=merge.param,
  export.merge.comp=T,
  ZNCcut=2,
  calcSW=FALSE,
  loadTree=FALSE,
  writeKME=TRUE,
  calcBigModStat=FALSE,
  writeModSnap=TRUE
)